# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaant7/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [8]:
import os
from dotenv import load_dotenv
import duckdb

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN is not None, "HF_TOKEN .env dosyasında bulunamadı"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Bucket by content age, compare decline rate
staleness_check = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    joined AS (
        SELECT d.*, c.content_created_date,
               DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
        FROM daily_agg d
        JOIN {TABLES['dim_content']} c ON d.content_hash_id = c.content_hash_id
        WHERE d.total_impressions > 0
    )
    SELECT
        CASE WHEN content_age_days >= 180 THEN 'stale (180+ days)' ELSE 'fresh (<180 days)' END AS age_bucket,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(CASE WHEN imp_second_half < imp_first_half THEN 1 ELSE 0 END) / COUNT(*), 1) AS decline_pct
    FROM joined
    GROUP BY age_bucket
""").df()
staleness_check

,age_bucket,n,decline_pct
0,stale (180+ days),93131,38.6
1,fresh (<180 days),83607,36.6


Signal 1 — Staleness (content_age_days ≥ 180): Bucketed by age (fresh vs. stale, n=83,607 / n=93,131), decline rate is 36.6% for fresh content vs. 38.6% for stale content — only a 2-point gap. The direction matches the assumption behind FlyRank's stale_visible_page flag (older content is riskier), but the effect is too weak to call it a strong signal on its own. Verdict: MIXED — direction confirmed, magnitude weak.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

baseline = con.sql(f"""
    WITH daily_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 500
    )
    SELECT
        content_hash_id,
        total_impressions,
        total_clicks,
        avg_position,
        100.0 * total_clicks / total_impressions AS ctr
    FROM daily_agg
    WHERE avg_position > 0 AND avg_position <= 20
""").df()

# Only keep pages under the CTR threshold — the actual rule condition
baseline = baseline[baseline["ctr"] < 0.5].copy()

# Score: reward high impressions + large CTR gap
baseline["baseline_score"] = baseline["total_impressions"] * (0.5 - baseline["ctr"]) / 0.5
baseline["reason_code"] = "low_ctr_visible_page"
baseline["action"] = "review_title_meta"

baseline = baseline.sort_values("baseline_score", ascending=False).reset_index(drop=True)
baseline["rank"] = baseline.index + 1

print("Queue shape:", baseline.shape)
baseline.head(10)

Queue shape: (41169, 9)


,content_hash_id,total_impressions,total_clicks,avg_position,ctr,baseline_score,reason_code,action,rank
0,content_44f34c0a90047651,212404.0,24.0,7.346909,0.011299,207604.0,low_ctr_visible_page,review_title_meta,1
1,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.142017,145697.0,low_ctr_visible_page,review_title_meta,2
2,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000741,134784.0,low_ctr_visible_page,review_title_meta,3
3,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.030066,134419.0,low_ctr_visible_page,review_title_meta,4
4,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000806,123875.0,low_ctr_visible_page,review_title_meta,5
5,content_b99ea6861864dea5,194337.0,361.0,4.450106,0.185760,122137.0,low_ctr_visible_page,review_title_meta,6
6,content_acbcc847f8996314,170808.0,262.0,3.361195,0.153389,118408.0,low_ctr_visible_page,review_title_meta,7
7,content_7c6373141eae744a,132593.0,83.0,5.789019,0.062598,115993.0,low_ctr_visible_page,review_title_meta,8
8,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,0.273138,111131.0,low_ctr_visible_page,review_title_meta,9
9,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.013943,104584.0,low_ctr_visible_page,review_title_meta,10


In [11]:
os.makedirs("../outputs", exist_ok=True)
baseline.to_csv("../outputs/baseline_action_score.csv", index=False)
print("Saved:", len(baseline), "rows")

Saved: 41169 rows


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

#1 (content_44f34c0a90047651): 212,404 impressions, position ~7.3, CTR 0.011% — near-zero clicks despite huge visibility and a strong position. Action: review_title_meta. Why: the largest score in the queue — massive visibility with essentially no clicks is the clearest possible signal of a title/snippet problem. What would make this wrong: if the SERP snippet is already fine and the real issue is a mismatched search intent (e.g. the page doesn't match what searchers actually want), a title rewrite wouldn't fix it — this would need an intent check, not just copy editing.

#2 (content_8d7d99f109e19aa2): 203,497 impressions, position ~2.6 (very strong), CTR 0.14%. Action: review_title_meta. Why: an excellent position with still-low CTR is a strong candidate — high visibility isn't the problem, something about the listing itself is. What would make this wrong: if this page has rich results (e.g. FAQ snippet, image pack) competing for attention on the SERP, the low CTR might reflect a crowded results page, not a bad title.

#3 (content_8e1334d6356668e3): 134,984 impressions, only 1 click total, CTR 0.0007%. Action: review_title_meta. Why: essentially zero clicks at high visibility — an extreme case. What would make this wrong: a CTR this close to zero across 134K impressions is unusual enough that I'd want to rule out a tracking/attribution issue before assuming it's purely a content problem.

#4 (content_34a70fea29d15f24): 143,019 impressions, position ~3.2, CTR 0.03%. Action: review_title_meta. Why: strong position, still very low CTR — matches the pattern. What would make this wrong: if this keyword's typical CTR at position 3 is naturally low for this intent type (e.g. informational queries with a featured snippet absorbing clicks), the "problem" might be structural, not fixable by title alone.

#5 (content_fec55986a1868d62): 124,075 impressions, 1 click, CTR 0.0008%. Action: review_title_meta. Why: same near-zero-click pattern as #3. What would make this wrong: same caveat — worth a quick tracking sanity check before treating this as a content issue.

#6 (content_b99ea6861864dea5): 194,337 impressions, position ~4.5, CTR 0.19%. Action: review_title_meta. Why: strong position, high volume, CTR still well under the 0.5% threshold. What would make this wrong: if this page recently changed its title (e.g. mid-March) and March's CTR reflects a stale old title in the impressions count, the fix might already be underway and this pick would be outdated.

#7 (content_acbcc847f8996314): 170,808 impressions, position ~3.4, CTR 0.15%. Action: review_title_meta. Why: same pattern — good position, high volume, weak CTR. What would make this wrong: same "already-fixed" caveat as #6, or a snippet that's being displayed differently than the raw title (e.g. Google rewriting it) making a title edit less impactful than expected.

#8 (content_7c6373141eae744a): 132,593 impressions, position ~5.8, CTR 0.06%. Action: review_title_meta. Why: consistent with the pattern — strong visibility, weak clicks. What would make this wrong: if competitor results changed mid-month (a new strong competitor entered the top 5), the CTR drop might be market-driven, not something a title fix alone resolves.

#9 (content_e8a52cf3d5988c07): 244,931 impressions, position ~15 (weaker than others in this list), CTR 0.27%. Action: review_title_meta. Why: highest raw impressions in the queue, but position is notably worse than #1–#8 — still flagged because the CTR gap is large in absolute terms. What would make this wrong: at position 15, low CTR is partly expected regardless of title quality — this pick may be more about position than about the listing itself, worth double-checking against the position-CTR baseline from Signal 2.

#10 (content_f6116743b00afc2d): 107,584 impressions, position ~9.5, CTR 0.014%. Action: review_title_meta. Why: strong position, very low CTR — clear candidate. What would make this wrong: same tracking-sanity caveat as #3/#5 — CTR this close to zero at meaningful impressions volume deserves a quick data-quality check before assuming it's a content issue.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weakest/most questionable picks in the top 10:

    #3, #5, #10 — all have 0-1 total clicks against 100K+ impressions (CTR essentially 0.0007%–0.014%). While this technically maximizes the score formula, a CTR this extreme is unusual enough to raise a data-quality question rather than a pure content-quality one: is this really "bad title," or could it be a tracking/attribution gap in how clicks are recorded for these specific pages? I'd want to spot-check these three against raw GSC data before trusting them at face value.
    
    #9 — ranked in the top 10 by score, but its actual position (~15) is meaningfully worse than the others (~2–9). Signal 2 showed CTR naturally drops with worse position — so part of #9's low CTR may simply be "normal for position 15," not a genuine title/snippet failure. The score formula weights raw impression volume heavily, which let a weaker-position page outscore some better-positioned, lower-volume pages. This is a small formula weakness worth noting: it doesn't fully separate "genuinely underperforming for its position" from "just has huge volume."

What I'd change if I revisited the rule: weight the CTR gap relative to the position-tier's typical CTR (an "expected CTR by position" baseline, as Lane 4 does), rather than a single flat 0.5% threshold for all positions — this would stop position-15 pages from competing unfairly against position-3 pages on the same scale.

Leakage check:

    No future-window inputs: all fields (total_impressions, total_clicks, avg_position, ctr) are aggregated only from March 2026 daily rows (month = '2026-03') — no data from April onward or from fact_daily_sample (the sealed June 2026 month) was touched.

    No label-derived inputs: this rule doesn't use any is_declining or trend label at all — it's a pure content/search signal rule (impressions, clicks, position), independent of the classification target from Week 2/3. There's nothing to leak from a label that isn't part of the rule.

    No product-context fields: the rule uses only gsc_impressions, gsc_clicks, gsc_avg_position — none of FlyRank's own decision flags (health_score, priority_score, action_type) are in the warehouse release at all, so there's nothing to accidentally reuse.

    dim_content fields not used in this rule: unlike Week 3's leakage hunt (which flagged word_count/backlinks as risky due to post-window updates), this rule doesn't touch dim_content at all — it's built entirely from within-window fact_daily aggregates, so the temporal leakage risk found last week doesn't apply here.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.